# Derived outputs

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/08-derived-outputs.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

[Model structures](./02-model-structures.ipynb) introduced compartments and flows.
Here we look at quantities **derived** from a run — not only compartment sizes,
but also flow masses and simple combinations of the two. In summer4 these are
requested through a {class}`~summer4.results.plan.SavePlan` with
{class}`~summer4.results.plan.Compartments`, {class}`~summer4.results.plan.FlowMass`,
and (when needed) host-side arithmetic on the resulting {class}`~summer4.results.output.Output`
objects.


In [ ]:
from typing import Any

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    ExitFlow,
    FlowMass,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    FlowModel,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

AXIS_COMP = {"index": "time", "value": "compartment size"}
AXIS_RATE = {"index": "time", "value": "rate per unit time"}


def make_y0(pmap: PropertyMap, state: Property, population: float, seed: float) -> np.ndarray:
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["susceptible"])] = population - seed
    y0[pmap.select(state["infectious"])] = seed
    return y0


def state_frame(res: Any, state: Property) -> pd.DataFrame:
    names = [t for t in state.traits]
    data = {
        name: np.asarray(res["comp"].select(state[name]).values.data).ravel()
        for name in names
    }
    return pd.DataFrame(data, index=np.asarray(res["comp"].times.values))


def series_total(output: Any) -> pd.Series:
    frame = output.total().to_pandas()
    return frame.iloc[:, 0]


In [ ]:
def get_sir_model() -> tuple[Any, PropertyMap, Property]:
    """SIR with frequency-dependent infection, recovery, and infection death."""
    state = Property("state", ("susceptible", "infectious", "recovered"))
    pop = Property("pop", ("all",))
    pmap = PropertyMap.from_property(state).stratify(pop)
    model = FlowModel(pmap)
    
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    model.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=pop,
    kind="frequency",
    contact_rate=Param("contact_rate"),
    mixing=mixing,
),
    )
)
    model.add_flow(TransitionFlow(
        "recovery",
        state["infectious"],
        state["recovered"],
        Param("recovery"),
    ))
    model.add_flow(ExitFlow("infection_death", state["infectious"], Param("infection_death")))
    return model.compile(), pmap, state


model_config = {"population": 1000.0, "seed": 10.0, "end_time": 20.0}
parameters = {"recovery": 0.333, "infection_death": 0.05, "contact_rate": 1.0}
times = np.linspace(0.0, model_config["end_time"], int(model_config["end_time"] * 10) + 1)

cm, pmap, state = get_sir_model()
y0 = make_y0(pmap, state, model_config["population"], model_config["seed"])
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)
res = cm.run(parameters, y0, t0=0.0, t1=model_config["end_time"], dt=0.1, save=plan, solver="dopri5")
compartment_values = state_frame(res, state)
assert float(compartment_values["infectious"].max()) > model_config["seed"]
compartment_values.plot.area(labels=AXIS_COMP, title="SIR compartments")


## Calculations based on compartment sizes

A useful derived quantity is the proportion of the population **ever infected**
— everyone currently infectious or recovered. That is a natural model analogue
of a serosurvey. Summer4 does not need a special request type for this: select
the relevant compartments from a saved `Compartments` output and normalise by the
total population.


In [ ]:
cm, pmap, state = get_sir_model()
y0 = make_y0(pmap, state, model_config["population"], model_config["seed"])
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)
res = cm.run(parameters, y0, t0=0.0, t1=model_config["end_time"], dt=0.1, save=plan, solver="dopri5")

ever = np.asarray(res["comp"].select(state["infectious"] | state["recovered"]).total().values).ravel()
total = np.asarray(res["comp"].total().values).ravel()
# Deaths shrink N; use live population in the denominator.
prop_ever = pd.Series(ever / total, index=times, name="prop_ever_infected")
assert float(prop_ever.iloc[-1]) > float(prop_ever.iloc[0])
assert float(prop_ever.max()) <= 1.0 + 1e-6
prop_ever.plot.area(
    labels={"index": "time", "value": "seropositive proportion"},
    title="Proportion ever infected",
).update_layout(showlegend=False)


## Flow outputs

Compartment sizes are prevalences. The **absolute** magnitude of a flow —
people transitioning per unit time — is an incidence-style quantity. In summer4
request {class}`~summer4.results.plan.FlowMass` for a named flow; `.total()`
sums over edges. Units are persons per unit time, not persons.

Examples we can form from this SIR model:

- new infections (`infection` flow)
- infection-specific deaths (`infection_death` exit)

![](figures/08/sir_transition.svg)


In [ ]:
cm, pmap, state = get_sir_model()
y0 = make_y0(pmap, state, model_config["population"], model_config["seed"])
plan = SavePlan(
    requests={
        "comp": SaveRequest(Compartments()),
        "incidence": SaveRequest(FlowMass(flow="infection")),
        "mortality": SaveRequest(FlowMass(flow="infection_death")),
    },
    ts=times,
)
res = cm.run(parameters, y0, t0=0.0, t1=model_config["end_time"], dt=0.1, save=plan, solver="dopri5")
sir_outputs = pd.DataFrame(
    {
        "incidence": series_total(res["incidence"]),
        "mortality": series_total(res["mortality"]),
    }
)
assert float(sir_outputs["incidence"].max()) > float(sir_outputs["mortality"].max())
sir_outputs.plot(labels=AXIS_RATE, title="SIR flow masses")


### Distinguishing infection from incidence

In an SIR model, infection and clinical incidence coincide because infectiousness
starts at the moment of infection. With a latent compartment (see
[series compartments and latency](./05-series-compartments-latency.ipynb)),
**infection** is the `infection` flow into `exposed`, while **incidence** of
infectious cases is better tracked on the `progression` flow into `infectious`.

![](figures/08/seir_transition.svg)


In [ ]:
def get_seir_model() -> tuple[Any, PropertyMap, Property]:
    """SEIR with infection into exposed and progression to infectious."""
    state = Property("state", ("susceptible", "exposed", "infectious", "recovered"))
    pop = Property("pop", ("all",))
    pmap = PropertyMap.from_property(state).stratify(pop)
    model = FlowModel(pmap)
    
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    model.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["exposed"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=pop,
    kind="frequency",
    contact_rate=Param("contact_rate"),
    mixing=mixing,
),
    )
)
    model.add_flow(TransitionFlow(
        "progression",
        state["exposed"],
        state["infectious"],
        Param("progression"),
    ))
    model.add_flow(TransitionFlow(
        "recovery",
        state["infectious"],
        state["recovered"],
        Param("recovery"),
    ))
    model.add_flow(ExitFlow("infection_death", state["infectious"], Param("infection_death")))
    return model.compile(), pmap, state


seir_parameters = {**parameters, "progression": 0.5}
cm, pmap, state = get_seir_model()
y0 = make_y0(pmap, state, model_config["population"], model_config["seed"])
plan = SavePlan(
    requests={
        "infection": SaveRequest(FlowMass(flow="infection")),
        "incidence": SaveRequest(FlowMass(flow="progression")),
    },
    ts=times,
)
res = cm.run(seir_parameters, y0, t0=0.0, t1=model_config["end_time"], dt=0.1, save=plan, solver="dopri5")
seir_outputs = pd.DataFrame(
    {
        "infection": series_total(res["infection"]),
        "incidence": series_total(res["incidence"]),
    }
)
# Latency delays incidence relative to infection early on.
assert float(seir_outputs["incidence"].iloc[0]) < float(seir_outputs["infection"].iloc[0])
assert float(seir_outputs["incidence"].max()) > 0.0
seir_outputs.plot(labels=AXIS_RATE, title="Infection vs incidence (SEIR)")


## Incomplete case detection

Surveillance rarely sees every incident episode. A simple observation model
scales incidence by a case-detection ratio (CDR). Summer4 has no separate
"derived output function" API for this chapter's purpose: multiply the saved
flow-mass series by the CDR on the host after the run (or fold the same product
into a calibration loss). Here CDR $= 0.5$.


In [ ]:
cdr = 0.5
notifications = seir_outputs["incidence"] * cdr
partial = pd.DataFrame(
    {
        "incidence": seir_outputs["incidence"],
        "notifications": notifications,
    }
)
assert np.allclose(partial["notifications"], 0.5 * partial["incidence"])
partial.plot(labels=AXIS_RATE, title="Incidence and notifications (CDR = 0.5)")


If calibration targets are notifications rather than true incidence, compare
data to this scaled series (or an equivalent `SaveFn` / loss construction), not
to the raw infection flow.
